# Epileptic Seizure Detection — CHB-MIT FAST TRAINING (5 patients)
# Uses only the seizure-rich patients for a quick presentation-ready model
# Expected runtime: 15–20 minutes

# STEP 0: SET YOUR DATA PATH
data_dir = r'E:\Epileptic Seizure\chb-mit-scalp-eeg-database-1.0.0'

import os
assert os.path.exists(data_dir), f"Data directory not found: {data_dir}"
print(f"✓ Data directory confirmed: {data_dir}")

In [18]:
data_dir = r'E:\Epileptic Seizure Dataset\chb-mit-scalp-eeg-database-1.0.0\chb-mit-scalp-eeg-database-1.0.0'

import os
assert os.path.exists(data_dir), f"Data directory not found: {data_dir}"
print(f"✓ Data directory confirmed: {data_dir}")

✓ Data directory confirmed: E:\Epileptic Seizure Dataset\chb-mit-scalp-eeg-database-1.0.0\chb-mit-scalp-eeg-database-1.0.0


# STEP 1: IMPORTS

In [19]:
import mne
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, f1_score, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import RandomizedSearchCV
from scipy import signal, stats
import joblib
import glob
import re
import gc
import os


print("✓ All libraries imported successfully.")
print(f"  TensorFlow version: {tf.__version__}")
print(f"  MNE version: {mne.__version__}")

✓ All libraries imported successfully.
  TensorFlow version: 2.21.0
  MNE version: 1.12.1


# STEP 2: CONFIGURATION

In [20]:
FS          = 256
WINDOW_SEC  = 5
WINDOW_SIZE = FS * WINDOW_SEC
MAX_WINDOWS_PER_FILE = 3000  # Reduced for speed

COMMON_CHANNELS = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FZ-CZ',  'CZ-PZ',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
]

CACHE_DIR = 'data_cache_fast'
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Config:")
print(f"  Sampling rate : {FS} Hz")
print(f"  Window size   : {WINDOW_SEC}s → {WINDOW_SIZE} samples")
print(f"  Channels      : {len(COMMON_CHANNELS)}")

Config:
  Sampling rate : 256 Hz
  Window size   : 5s → 1280 samples
  Channels      : 18


# STEP 3: SELECT ONLY 5 SEIZURE-RICH PATIENTS FOR SPEED

In [21]:
# These 5 patients have clear, unambiguous seizures
TARGET_PATIENTS = ['chb01', 'chb03', 'chb06', 'chb08', 'chb12']

edf_files_all = sorted(glob.glob(os.path.join(data_dir, '*', '*.edf')))
edf_files = [f for f in edf_files_all 
             if any(p in os.path.basename(f) for p in TARGET_PATIENTS)]

print(f"\n✓ Selected {len(edf_files)} EDF files from {len(TARGET_PATIENTS)} patients")
for p in TARGET_PATIENTS:
    count = len([f for f in edf_files if p in os.path.basename(f)])
    print(f"  {p}: {count} files")


✓ Selected 142 EDF files from 5 patients
  chb01: 42 files
  chb03: 38 files
  chb06: 18 files
  chb08: 20 files
  chb12: 24 files


# STEP 4: PARSE SEIZURE REGISTRY

In [22]:
def parse_seizure_registry(data_dir):
    seizure_registry = {}
    summary_files = glob.glob(os.path.join(data_dir, '*', '*-summary.txt'))

    for summary_file in sorted(summary_files):
        with open(summary_file, 'r') as f:
            lines = f.readlines()

        current_file  = None
        pending_start = None

        for line in lines:
            if 'File Name:' in line:
                m = re.search(r'File Name:\s+(\S+\.edf)', line, re.IGNORECASE)
                if m:
                    current_file = m.group(1)
            elif 'Seizure' in line and 'Start Time:' in line:
                m = re.search(r'Start Time:\s+(\d+)\s+seconds', line)
                if m:
                    pending_start = int(m.group(1))
            elif 'Seizure' in line and 'End Time:' in line:
                m = re.search(r'End Time:\s+(\d+)\s+seconds', line)
                if m and pending_start is not None and current_file is not None:
                    seizure_registry.setdefault(current_file, []).append(
                        (pending_start, int(m.group(1)))
                    )
                    pending_start = None

    total = sum(len(v) for v in seizure_registry.values())
    print(f"✓ Parsed {len(seizure_registry)} files with seizures ({total} total)")
    return seizure_registry

seizure_registry = parse_seizure_registry(data_dir)

✓ Parsed 141 files with seizures (198 total)


# STEP 5: EXTRACT WINDOWS (MEMORY-SAFE)

In [23]:
def get_labeled_windows(file_path, is_seizure_file=False, seizure_registry=None):
    try:
        raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
        valid_picks = [ch for ch in COMMON_CHANNELS if ch in raw.ch_names]
        if not valid_picks:
            return np.zeros((0, WINDOW_SIZE * len(COMMON_CHANNELS)), np.float32), np.array([])
        raw.pick(valid_picks)
        raw.load_data()  # Load data into memory after picking channels
        raw.filter(l_freq=0.5, h_freq=40.0, method='fir', verbose=False)

        n_samples  = raw.n_times
        n_ch       = len(raw.ch_names)
        fname      = os.path.basename(file_path)
        windows, labels = [], []

        if is_seizure_file and seizure_registry and fname in seizure_registry:
            for start_s, end_s in seizure_registry[fname]:
                start_sample = int(start_s * FS)
                end_sample   = min(int(end_s * FS), n_samples)
                for i in range(start_sample, end_sample - WINDOW_SIZE, WINDOW_SIZE):
                    data, _ = raw[:, i : i + WINDOW_SIZE]
                    windows.append(data.flatten().astype(np.float32))
                    labels.append(1)
        else:
            count = 0
            for i in range(0, n_samples - WINDOW_SIZE, WINDOW_SIZE):
                if count >= MAX_WINDOWS_PER_FILE:
                    break
                data, _ = raw[:, i : i + WINDOW_SIZE]
                windows.append(data.flatten().astype(np.float32))
                labels.append(0)
                count += 1

        del raw
        gc.collect()

        if windows:
            return np.array(windows, dtype=np.float32), np.array(labels, dtype=np.int8)
        return np.zeros((0, WINDOW_SIZE * n_ch), np.float32), np.array([])

    except Exception as e:
        print(f"  ✗ Error on {os.path.basename(file_path)}: {e}")
        return np.zeros((0, WINDOW_SIZE * len(COMMON_CHANNELS)), np.float32), np.array([])

print("✓ Window extraction function ready")

✓ Window extraction function ready


# STEP 6: LOAD ALL 5 PATIENTS (NO MEMMAP, DIRECT ARRAYS)

In [ ]:
print(f"\nLoading {len(edf_files)} EDF files...")
all_X = []
all_y = []
file_group = []
file_idx = 0
MAX_SAMPLES_PER_FILE = 300  # Significantly reduced for memory efficiency

for file_path in edf_files:
    fname = os.path.basename(file_path)
    is_seizure = fname in seizure_registry
    
    X_win, y_win = get_labeled_windows(file_path, is_seizure_file=is_seizure, 
                                       seizure_registry=seizure_registry)
    
    if len(X_win) > 0:
        # Subsample if too many windows
        if len(X_win) > MAX_SAMPLES_PER_FILE:
            # Keep all seizure samples, subsample normal samples
            if y_win.sum() > 0:
                seizure_idx = np.where(y_win == 1)[0]
                normal_idx = np.where(y_win == 0)[0]
                num_normal = MAX_SAMPLES_PER_FILE - len(seizure_idx)
                if num_normal > 0 and len(normal_idx) > num_normal:
                    normal_idx = np.random.choice(normal_idx, num_normal, replace=False)
                    indices = np.concatenate([seizure_idx, normal_idx])
                else:
                    indices = seizure_idx[:MAX_SAMPLES_PER_FILE]
            else:
                indices = np.random.choice(len(X_win), MAX_SAMPLES_PER_FILE, replace=False)
            X_win = X_win[indices]
            y_win = y_win[indices]
        
        all_X.append(X_win)
        all_y.append(y_win)
        file_group.extend([file_idx] * len(y_win))
        print(f"  ✓ {fname}: {len(y_win)} windows ({int(y_win.sum())} seizure)")
        file_idx += 1
    else:
        print(f"  ✗ {fname}: no windows")
    
    del X_win, y_win
    gc.collect()

# Concatenate all data
if all_X:
    X = np.concatenate(all_X, axis=0).astype(np.float32)
    y = np.concatenate(all_y, axis=0).astype(np.int8)
    groups = np.array(file_group, dtype=np.int32)
    
    print(f"\n✓ Dataset loaded:")
    print(f"  X shape      : {X.shape}")
    print(f"  Memory usage : ~{X.nbytes / 1e9:.2f} GB")
    print(f"  Normal       : {int((y==0).sum()):,}")
    print(f"  Seizure      : {int((y==1).sum()):,}")
    print(f"  Unique files : {len(np.unique(groups))}")
    
    # Clear intermediate lists
    del all_X, all_y, file_group
    gc.collect()
else:
    print("\n✗ No data was loaded. Halting execution.")
    assert False, "No windows were extracted from the EDF files."


Loading 142 EDF files...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


C:\Users\asada\AppData\Local\Temp\ipykernel_1456\3337757615.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)


  ✓ chb01_01.edf: 719 windows (0 seizure)
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


C:\Users\asada\AppData\Local\Temp\ipykernel_1456\3337757615.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)


  ✓ chb01_02.edf: 719 windows (0 seizure)
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


...

C:\Users\asada\AppData\Local\Temp\ipykernel_1456\3337757615.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
C:\Users\asada\AppData\Local\Temp\ipykernel_1456\3337757615.py:3: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)


  ✓ chb12_41.edf: 719 windows (0 seizure)
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


C:\Users\asada\AppData\Local\Temp\ipykernel_1456\3337757615.py:3: RuntimeWarning: Channel names are not unique, found duplicates for: {'-', 'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
C:\Users\asada\AppData\Local\Temp\ipykernel_1456\3337757615.py:3: RuntimeWarning: Scaling factor is not defined in following channels:
--0, --1, --2, --3, --4
  raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)


  ✓ chb12_42.edf: 28 windows (28 seizure)


MemoryError: Unable to allocate 6.20 GiB for an array with shape (76523, 21760) and data type float32

# STEP 7: TRAIN/TEST SPLIT

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_raw = X[train_idx].copy()
y_train     = y[train_idx].copy().astype(int)
X_test_raw  = X[test_idx].copy()
y_test      = y[test_idx].copy().astype(int)

# Delete large original arrays
del X, y, groups
gc.collect()

# Scale the data
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

# Clean up raw data
del X_train_raw, X_test_raw
gc.collect()

joblib.dump(scaler, 'scaler.pkl')

print(f"\nTrain/Test split:")
print(f"  Train : {len(y_train):,}  (Normal={int((y_train==0).sum()):,}, Seizure={int((y_train==1).sum()):,})")
print(f"  Test  : {len(y_test):,}  (Normal={int((y_test==0).sum()):,}, Seizure={int((y_test==1).sum()):,})")
print(f"✓ scaler.pkl saved")

MemoryError: Unable to allocate 6.32 GiB for an array with shape (78002, 21760) and data type float32

# STEP 8: FEATURE EXTRACTION

In [ ]:
N_CHANNELS = X_train_scaled.shape[1] // WINDOW_SIZE

def extract_features_batch(X_flat, window_size=WINDOW_SIZE, n_ch=N_CHANNELS, sfreq=FS):
    n_windows = X_flat.shape[0]
    X_3d = X_flat.reshape(n_windows, window_size, n_ch)
    features = []

    for i in range(n_windows):
        win_feats = []
        for ch in range(n_ch):
            ch_data = X_3d[i, :, ch].astype(np.float64)

            # Time-domain
            win_feats += [
                ch_data.mean(),
                ch_data.std(),
                ch_data.var(),
                np.percentile(ch_data, 25),
                np.percentile(ch_data, 75),
                float(stats.skew(ch_data)),
                float(stats.kurtosis(ch_data)),
                ch_data.min(),
                ch_data.max(),
            ]

            # Band power
            freqs, psd = signal.welch(ch_data, sfreq, nperseg=min(sfreq, window_size))
            win_feats += [
                float(psd[(freqs >= 0.5) & (freqs < 4 )].mean()),
                float(psd[(freqs >= 4  ) & (freqs < 8 )].mean()),
                float(psd[(freqs >= 8  ) & (freqs < 13)].mean()),
                float(psd[(freqs >= 13 ) & (freqs < 30)].mean()),
                float(psd[(freqs >= 30 ) & (freqs < 50)].mean()),
            ]
        features.append(win_feats)

    return np.array(features, dtype=np.float32)

print(f"Extracting features...")
X_train_feat = extract_features_batch(X_train_scaled)
X_test_feat  = extract_features_batch(X_test_scaled)

print(f"  X_train_feat : {X_train_feat.shape}")
print(f"  X_test_feat  : {X_test_feat.shape}")

# STEP 9: CLASS WEIGHTS

In [ ]:
class_weights_arr = compute_class_weight('balanced',
                                          classes=np.unique(y_train),
                                          y=y_train)
class_weights_dict = {int(cls): float(w)
                      for cls, w in zip(np.unique(y_train), class_weights_arr)}

print(f"Class weights: {class_weights_dict}")

# STEP 10: TRAIN RANDOM FOREST BASELINE

In [ ]:
print(f"\nTraining Random Forest...")
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
rf_baseline.fit(X_train_feat, y_train)

rf_test_pred = rf_baseline.predict(X_test_feat)
rf_test_prob = rf_baseline.predict_proba(X_test_feat)[:, 1]
rf_train_acc = accuracy_score(y_train, rf_baseline.predict(X_train_feat))
rf_test_acc  = accuracy_score(y_test, rf_test_pred)
rf_f1        = f1_score(y_test, rf_test_pred, zero_division=0)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_test_prob)
rf_auc       = auc(rf_fpr, rf_tpr)

print(f"\n✓ Random Forest Baseline:")
print(f"  Train accuracy : {rf_train_acc:.4f}")
print(f"  Test accuracy  : {rf_test_acc:.4f}")
print(f"  F1-score       : {rf_f1:.4f}")
print(f"  ROC-AUC        : {rf_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, rf_test_pred,
      target_names=['Normal', 'Seizure'], zero_division=0))

# STEP 11: RF HYPERPARAMETER TUNING

In [ ]:
print(f"\nTuning Random Forest (RandomizedSearchCV)...")
param_dist = {
    'n_estimators'    : [100, 200],
    'max_depth'       : [10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf' : [1, 2],
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=5,
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1,
)
rf_search.fit(X_train_feat, y_train)

best_rf = rf_search.best_estimator_
best_rf_pred = best_rf.predict(X_test_feat)
best_rf_prob = best_rf.predict_proba(X_test_feat)[:, 1]
best_rf_acc  = accuracy_score(y_test, best_rf_pred)
best_rf_f1   = f1_score(y_test, best_rf_pred, zero_division=0)
best_rf_fpr, best_rf_tpr, _ = roc_curve(y_test, best_rf_prob)
best_rf_auc  = auc(best_rf_fpr, best_rf_tpr)

print(f"\n✓ Tuned Random Forest:")
print(f"  Best params    : {rf_search.best_params_}")
print(f"  Test accuracy  : {best_rf_acc:.4f}")
print(f"  F1-score       : {best_rf_f1:.4f}")
print(f"  ROC-AUC        : {best_rf_auc:.4f}")
print(classification_report(y_test, best_rf_pred,
      target_names=['Normal', 'Seizure'], zero_division=0))

joblib.dump(best_rf, 'best_rf_tuned_model.pkl')
print("\n✅ best_rf_tuned_model.pkl saved")

# STEP 12: RESHAPE FOR 1D-CNN

In [ ]:
N_CH_CNN = X_train_scaled.shape[1] // WINDOW_SIZE

X_train_cnn = X_train_scaled.reshape(-1, WINDOW_SIZE, N_CH_CNN)
X_test_cnn  = X_test_scaled.reshape(-1, WINDOW_SIZE, N_CH_CNN)

print(f"\nCNN input shapes:")
print(f"  X_train_cnn : {X_train_cnn.shape}")
print(f"  X_test_cnn  : {X_test_cnn.shape}")

# STEP 13: BUILD 1D-CNN

In [ ]:
def build_cnn(input_shape):
    model = models.Sequential([
        layers.Conv1D(16, kernel_size=3, activation='relu',
                      input_shape=input_shape,
                      kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(32, kernel_size=3, activation='relu',
                      kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),

        layers.Conv1D(64, kernel_size=3, activation='relu',
                      kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),

        layers.Dense(64, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc'),
                 tf.keras.metrics.Recall(name='sensitivity')],
    )
    return model

print(f"\nBuilding 1D-CNN...")
advanced_model = build_cnn((WINDOW_SIZE, N_CH_CNN))
advanced_model.summary()

# STEP 14: TRAIN 1D-CNN

In [ ]:
print(f"\nTraining 1D-CNN (this takes ~5-10 min)...")
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1),
]

history = advanced_model.fit(
    X_train_cnn, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1,
)

# STEP 15: EVALUATE 1D-CNN

In [ ]:
cnn_test_prob = advanced_model.predict(X_test_cnn, verbose=0).flatten()
cnn_test_pred = (cnn_test_prob > 0.5).astype(int)

cnn_train_pred = (advanced_model.predict(X_train_cnn, verbose=0).flatten() > 0.5).astype(int)
cnn_train_acc = accuracy_score(y_train, cnn_train_pred)
cnn_test_acc  = accuracy_score(y_test,  cnn_test_pred)
cnn_f1        = f1_score(y_test, cnn_test_pred, zero_division=0)
cnn_fpr, cnn_tpr, _ = roc_curve(y_test, cnn_test_prob)
cnn_auc       = auc(cnn_fpr, cnn_tpr)

print(f"\n✓ 1D-CNN Results:")
print(f"  Train accuracy : {cnn_train_acc:.4f}")
print(f"  Test accuracy  : {cnn_test_acc:.4f}")
print(f"  F1-score       : {cnn_f1:.4f}")
print(f"  ROC-AUC        : {cnn_auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, cnn_test_pred,
      target_names=['Normal', 'Seizure'], zero_division=0))

# STEP 16: SAVE EVERYTHING

In [ ]:
advanced_model.save('advanced_model.h5')
joblib.dump(history.history, 'training_history.pkl')

metrics = {
    'dataset_info': {
        'total_windows'  : int(len(y)),
        'normal_samples' : int((y == 0).sum()),
        'seizure_samples': int((y == 1).sum()),
        'num_files'      : len(np.unique(groups)),
        'data_source'    : 'CHB-MIT (5 patients)',
    },
    'train_test_split': {
        'train_samples': int(len(y_train)),
        'test_samples' : int(len(y_test)),
        'split_method' : 'GroupShuffleSplit (patient-aware)',
    },
    'random_forest': {
        'test_acc' : float(best_rf_acc),
        'f1_score' : float(best_rf_f1),
        'roc_auc'  : float(best_rf_auc),
    },
    'cnn_1d': {
        'test_acc' : float(cnn_test_acc),
        'f1_score' : float(cnn_f1),
        'roc_auc'  : float(cnn_auc),
    },
}
joblib.dump(metrics, 'model_metrics.pkl')

print("\n✅ All files saved:")
print("   scaler.pkl")
print("   best_rf_tuned_model.pkl")
print("   advanced_model.h5")
print("   model_metrics.pkl")
print(f"\n🎉 TRAINING COMPLETE! Ready for app.py")